# 05 - Monitoreo e incidentes — versión de clase

## Objetivos de aprendizaje

- Comparar escenarios sin cambiar las reglas del contrato.
- Interpretar tasas con sus denominadores y distinguir filas de hallazgos.
- Detectar deriva antes de transformar y registrar su evidencia separada.
- Seguir un incidente desde su asignación hasta una recuperación verificada.

SQLite funcionará **en memoria**: al cerrar la conexión se pierden tablas y auditoría. No hay alertas externas ni identidades autenticadas. El Steward investiga; el Owner autoriza; el custodio ejecuta y conserva evidencia. Aquí son roles informativos.

## Cómo trabajar

Ejecuta **una celda por vez con `Shift+Enter`**, de arriba hacia abajo. Antes de avanzar, observa la salida y responde la pregunta de la sección. Las funciones aparecen donde se necesitan, no se ejecuta todo al final.

Solo necesitas Python, pandas y un kernel Jupyter/IPython. Todos los datos y funciones se crean aquí: no se importan archivos de código, no se leen archivos de entrada y no se requieren servicios ni Internet. Las comprobaciones se hacen con Python, sin pytest.

Usamos únicamente datos ficticios y dominios `.invalid`. **No los sustituyas por datos personales reales.** La fecha de corte es fija para obtener resultados reproducibles.

Para repetir la práctica, **reinicia el kernel y ejecuta desde el paso 1**. Repetir celdas aisladas puede modificar claves, variables o historial de intentos.


## 1. Preparar el entorno

Importamos las bibliotecas y fijamos la versión del contrato y la fecha de corte. `display` permite ver tablas dentro del notebook.

**Salida esperada:** Versión de pandas, contrato 1.0.0 y corte 2026-10-06 a las 12:00 UTC.

**Para conversar:** ¿Por qué una práctica reproducible no debe depender de la fecha de hoy?


In [1]:
import hashlib
import json
import sqlite3
from uuid import uuid4

import re
from decimal import Decimal, InvalidOperation

import pandas as pd
from IPython.display import display

FECHA_CORTE = pd.Timestamp("2026-10-06T12:00:00Z")
VERSION_CONTRATO = "1.0.0"
COLUMNAS_FUENTE = {
    "fila_fuente", "id_venta", "fecha_venta", "id_producto",
    "monto", "moneda", "email", "actualizado_en",
}

print("pandas:", pd.__version__)
print("Contrato:", VERSION_CONTRATO)
print("Fecha de corte:", FECHA_CORTE)


pandas: 3.0.2
Contrato: 1.0.0
Fecha de corte: 2026-10-06 12:00:00+00:00


## 2. Crear y observar las fuentes ficticias

Creamos ventas, maestro de productos y evidencia parcial de exactitud. La referencia es una tabla independiente para comparar importes, no una verdad conocida para todas las ventas.

**Salida esperada:** 12 ventas, 2 productos y 2 importes de referencia.

**Para conversar:** ¿Qué diferencia hay entre `fila_fuente` e `id_venta`? Busca un ID duplicado.


In [2]:
def crear_datos():
    """Retorna ventas ficticias, maestro y evidencia parcial independiente."""
    filas = [
        ("v01", "1", "2026-10-05", "101", "1500.00", " ars ", "uno@demo.invalid", "2026-10-06T10:00:00Z"),
        ("v02", "2", "2026-10-05", "102", "200.00", "ARS", "dos@demo.invalid", "2026-10-06T10:00:00Z"),
        ("v03", "2", "2026-10-05", "102", "250.00", "ARS", "dos@demo.invalid", "2026-10-06T10:00:00Z"),
        ("v04", "3", "2026-02-29", "101", "300.00", "ARS", "tres@demo.invalid", "2026-10-06T10:00:00Z"),
        ("v05", "4", "2026-10-05", "101", "-10.00", "ARS", "cuatro@demo.invalid", "2026-10-06T10:00:00Z"),
        ("v06", "5", "2026-10-05", "101", "500.00", "BTC", "cinco@demo.invalid", "2026-10-06T10:00:00Z"),
        ("v07", "6", "2026-10-05", "999", "600.00", "ARS", "seis@demo.invalid", "2026-10-06T10:00:00Z"),
        ("v08", "7", "2026-10-05", "102", "700.00", "USD", "email-invalido", "2026-10-06T10:00:00Z"),
        ("v09", "8", "2026-09-30", "102", "800.00", "ARS", " ", "2026-10-01T10:00:00Z"),
        ("v10", "9", "2026-10-07", "101", "900.00", "ARS", None, "2026-10-06T10:00:00Z"),
        ("v11", "10", "2026-10-05", "101", "1000.00", "ARS", "diez@demo.invalid", "2026-10-06T10:00:00Z"),
        ("v12", " ", "2026-10-05", "101", "1200.00", "ARS", None, "2026-10-06T10:00:00Z"),
    ]
    ventas = pd.DataFrame(filas, columns=[
        "fila_fuente", "id_venta", "fecha_venta", "id_producto", "monto",
        "moneda", "email", "actualizado_en",
    ])
    productos = pd.DataFrame({"id_producto": [101, 102], "producto": ["Producto A", "Producto B"]})
    referencia = pd.DataFrame({"id_venta": [1, 10], "monto_centavos_referencia": [150000, 99900]})
    return ventas, productos, referencia

crudos, productos, referencia = crear_datos()
crudos_preservados = crudos.copy(deep=True)
print("Ventas de entrada:", len(crudos))
display(crudos)
print("Maestro de productos:")
display(productos)
print("Evidencia parcial de exactitud:")
display(referencia)
assert len(crudos) == 12 and len(referencia) == 2


Ventas de entrada: 12


,fila_fuente,id_venta,fecha_venta,id_producto,monto,moneda,email,actualizado_en
0,v01,1,2026-10-05,101,1500.00,ars,uno@demo.invalid,2026-10-06T10:00:00Z
1,v02,2,2026-10-05,102,200.00,ARS,dos@demo.invalid,2026-10-06T10:00:00Z
2,v03,2,2026-10-05,102,250.00,ARS,dos@demo.invalid,2026-10-06T10:00:00Z
3,v04,3,2026-02-29,101,300.00,ARS,tres@demo.invalid,2026-10-06T10:00:00Z
4,v05,4,2026-10-05,101,-10.00,ARS,cuatro@demo.invalid,2026-10-06T10:00:00Z
5,v06,5,2026-10-05,101,500.00,BTC,cinco@demo.invalid,2026-10-06T10:00:00Z
6,v07,6,2026-10-05,999,600.00,ARS,seis@demo.invalid,2026-10-06T10:00:00Z
7,v08,7,2026-10-05,102,700.00,USD,email-invalido,2026-10-06T10:00:00Z
8,v09,8,2026-09-30,102,800.00,ARS,,2026-10-01T10:00:00Z
9,v10,9,2026-10-07,101,900.00,ARS,NaN,2026-10-06T10:00:00Z


Maestro de productos:


,id_producto,producto
0,101,Producto A
1,102,Producto B


Evidencia parcial de exactitud:


,id_venta,monto_centavos_referencia
0,1,150000
1,10,99900


## 3. Normalizar sin modificar el origen

Primero validamos el esquema. Luego convertimos IDs a enteros anulables, fechas a UTC, importes a centavos con `Decimal` y monedas a mayúsculas. Una fecha imposible queda como `NaT`; no inventamos una fecha válida. Normalizar no significa aprobar los datos.

**Salida esperada:** Tabla normalizada con `monto_centavos`; el origen permanece igual.

**Para conversar:** ¿Por qué no conviene calcular dinero usando números de punto flotante?


In [3]:
def esquema_valido(datos):
    return set(datos.columns) == COLUMNAS_FUENTE



def centavos(valor):
    try:
        numero = Decimal(str(valor).strip())
        if not numero.is_finite() or numero != numero.quantize(Decimal("0.01")):
            return None
        return int(numero * 100)
    except (InvalidOperation, ValueError, TypeError):
        return None



def entero_positivo(valor):
    texto = str(valor).strip()
    return int(texto) if re.fullmatch(r"[1-9][0-9]*", texto) else None



def normalizar(datos):
    if not esquema_valido(datos):
        raise ValueError("Deriva de esquema: revisar contrato antes de transformar")
    if datos["fila_fuente"].isna().any() or datos["fila_fuente"].duplicated().any():
        raise ValueError("La identificacion tecnica de origen debe ser unica y no nula")
    resultado = datos.copy(deep=True)
    for columna in ["id_venta", "id_producto"]:
        resultado[columna] = pd.array(resultado[columna].map(entero_positivo), dtype="Int64")
    resultado["monto_centavos"] = pd.array(resultado["monto"].map(centavos), dtype="Int64")
    resultado["fecha_venta"] = pd.to_datetime(resultado["fecha_venta"], format="%Y-%m-%d", errors="coerce", utc=True)
    resultado["actualizado_en"] = pd.to_datetime(resultado["actualizado_en"], format="ISO8601", errors="coerce", utc=True)
    resultado["moneda"] = resultado["moneda"].astype("string").str.strip().str.upper()
    resultado["email"] = resultado["email"].astype("string").str.strip().replace("", pd.NA)
    return resultado

datos = normalizar(crudos)
display(datos[[
    "fila_fuente", "id_venta", "fecha_venta",
    "monto_centavos", "moneda", "email",
]])
pd.testing.assert_frame_equal(crudos, crudos_preservados)
print("El origen se conserva sin cambios.")


,fila_fuente,id_venta,fecha_venta,monto_centavos,moneda,email
0,v01,1,2026-10-05 00:00:00+00:00,150000,ARS,uno@demo.invalid
1,v02,2,2026-10-05 00:00:00+00:00,20000,ARS,dos@demo.invalid
2,v03,2,2026-10-05 00:00:00+00:00,25000,ARS,dos@demo.invalid
3,v04,3,NaT,30000,ARS,tres@demo.invalid
4,v05,4,2026-10-05 00:00:00+00:00,-1000,ARS,cuatro@demo.invalid
5,v06,5,2026-10-05 00:00:00+00:00,50000,BTC,cinco@demo.invalid
6,v07,6,2026-10-05 00:00:00+00:00,60000,ARS,seis@demo.invalid
7,v08,7,2026-10-05 00:00:00+00:00,70000,USD,email-invalido
8,v09,8,2026-09-30 00:00:00+00:00,80000,ARS,<NA>
9,v10,9,2026-10-07 00:00:00+00:00,90000,ARS,<NA>


El origen se conserva sin cambios.


## 4. Evaluar calidad y separar filas

Cada regla tiene dimensión y acción. Las fallas de cuarentena rechazan la fila; las advertencias no la rechazan. Una fila puede generar varios hallazgos. En exactitud solo se evalúan las filas que tienen referencia: no evaluable no significa correcto.

La función devuelve cuatro objetos: aceptadas, rechazadas, métricas y detalle. Observa cómo las máscaras de cada regla generan el detalle y cómo se reúnen los identificadores rechazados.

**Salida esperada:** 4 filas aptas, 8 rechazadas y 17 hallazgos; 2 filas evaluables en exactitud.

**Para conversar:** ¿Puede una fila apta conservar una advertencia de email o exactitud?


In [4]:
def evaluar(datos, productos, referencia):
    """Evalua todas las filas; exactitud solo sobre evidencia disponible."""
    referencia = referencia.copy()
    if referencia["id_venta"].duplicated().any() or referencia["id_venta"].isna().any():
        raise ValueError("La referencia de exactitud tiene claves invalidas")
    if productos["id_producto"].duplicated().any() or productos["id_producto"].isna().any():
        raise ValueError("El maestro de productos tiene claves invalidas")
    verdad = datos["id_venta"].map(referencia.set_index("id_venta")["monto_centavos_referencia"])
    edad_horas = (FECHA_CORTE - datos["actualizado_en"]).dt.total_seconds() / 3600
    reglas = [
        ("DQ_ID", "completitud", "cuarentena", datos["id_venta"].notna(), None),
        ("DQ_UNICIDAD", "unicidad", "cuarentena", ~datos["id_venta"].duplicated(keep=False) & datos["id_venta"].notna(), None),
        ("DQ_FECHA", "validez", "cuarentena", datos["fecha_venta"].notna() & datos["fecha_venta"].le(FECHA_CORTE), None),
        ("DQ_MONTO", "validez", "cuarentena", datos["monto_centavos"].notna() & datos["monto_centavos"].gt(0), None),
        ("DQ_MONEDA", "integridad", "cuarentena", datos["moneda"].isin(["ARS", "USD", "EUR"]), None),
        ("DQ_FK_PRODUCTO", "integridad", "cuarentena", datos["id_producto"].isin(productos["id_producto"]), None),
        ("DQ_ACTUALIZACION", "consistencia", "cuarentena", datos["actualizado_en"].notna() & datos["actualizado_en"].le(FECHA_CORTE) & datos["actualizado_en"].ge(datos["fecha_venta"]), None),
        ("DQ_EMAIL", "validez", "advertencia", datos["email"].str.fullmatch(r"[^\s@]+@[^\s@]+\.[^\s@]+").fillna(False), None),
        ("DQ_VIGENCIA", "vigencia", "advertencia", edad_horas.between(0, 48), None),
        ("DQ_EXACTITUD", "exactitud", "advertencia", datos["monto_centavos"].eq(verdad), verdad.notna()),
    ]
    detalle, resumen = [], []
    for regla_id, dimension, accion, cumple, elegible in reglas:
        elegible = pd.Series(True, index=datos.index) if elegible is None else elegible
        cumple = cumple.fillna(False)
        falla = elegible & ~cumple
        evaluados = int(elegible.sum())
        fallidos = int(falla.sum())
        resumen.append({
            "regla": regla_id, "dimension": dimension, "accion": accion,
            "evaluados": evaluados, "fallidos": fallidos,
            "no_evaluables": len(datos) - evaluados,
            "cumplimiento_pct": 100 * (evaluados - fallidos) / evaluados if evaluados else None,
            "owner": "Gerencia Comercial", "contrato": VERSION_CONTRATO,
        })
        for fila_fuente in datos.loc[falla, "fila_fuente"]:
            detalle.append({"fila_fuente": fila_fuente, "regla": regla_id, "accion": accion})
    detalle = pd.DataFrame(detalle, columns=["fila_fuente", "regla", "accion"])
    ids_rechazados = set(detalle.loc[detalle["accion"].eq("cuarentena"), "fila_fuente"])
    rechazadas = datos[datos["fila_fuente"].isin(ids_rechazados)].copy()
    aceptadas = datos[~datos["fila_fuente"].isin(ids_rechazados)].copy()
    assert len(datos) == len(aceptadas) + len(rechazadas)
    return aceptadas, rechazadas, pd.DataFrame(resumen), detalle

aceptadas, rechazadas, metricas, detalle = evaluar(datos, productos, referencia)
display(metricas)
display(detalle)
print("Aptas:", len(aceptadas), "| Rechazadas:", len(rechazadas))
display(aceptadas[["fila_fuente", "id_venta", "monto_centavos", "moneda"]])
assert (len(aceptadas), len(rechazadas), len(detalle)) == (4, 8, 17)
assert int(metricas.set_index("regla").loc["DQ_EXACTITUD", "evaluados"]) == 2


,regla,dimension,accion,evaluados,fallidos,no_evaluables,cumplimiento_pct,owner,contrato
0,DQ_ID,completitud,cuarentena,12,1,0,91.666667,Gerencia Comercial,1.0.0
1,DQ_UNICIDAD,unicidad,cuarentena,12,3,0,75.000000,Gerencia Comercial,1.0.0
2,DQ_FECHA,validez,cuarentena,12,2,0,83.333333,Gerencia Comercial,1.0.0
3,DQ_MONTO,validez,cuarentena,12,1,0,91.666667,Gerencia Comercial,1.0.0
4,DQ_MONEDA,integridad,cuarentena,12,1,0,91.666667,Gerencia Comercial,1.0.0
5,DQ_FK_PRODUCTO,integridad,cuarentena,12,1,0,91.666667,Gerencia Comercial,1.0.0
6,DQ_ACTUALIZACION,consistencia,cuarentena,12,2,0,83.333333,Gerencia Comercial,1.0.0
7,DQ_EMAIL,validez,advertencia,12,4,0,66.666667,Gerencia Comercial,1.0.0
8,DQ_VIGENCIA,vigencia,advertencia,12,1,0,91.666667,Gerencia Comercial,1.0.0
9,DQ_EXACTITUD,exactitud,advertencia,2,1,10,50.000000,Gerencia Comercial,1.0.0


,fila_fuente,regla,accion
0,v12,DQ_ID,cuarentena
1,v02,DQ_UNICIDAD,cuarentena
2,v03,DQ_UNICIDAD,cuarentena
3,v12,DQ_UNICIDAD,cuarentena
4,v04,DQ_FECHA,cuarentena
5,v10,DQ_FECHA,cuarentena
6,v05,DQ_MONTO,cuarentena
7,v06,DQ_MONEDA,cuarentena
8,v07,DQ_FK_PRODUCTO,cuarentena
9,v04,DQ_ACTUALIZACION,cuarentena


Aptas: 4 | Rechazadas: 8


,fila_fuente,id_venta,monto_centavos,moneda
0,v01,1,150000,ARS
7,v08,7,70000,USD
8,v09,8,80000,ARS
10,v11,10,100000,ARS


## 5. Preparar escenarios comparables

A partir de las cuatro filas aptas creamos un subconjunto de laboratorio. Después alteramos un solo importe a cero **en una copia**. Conservamos las fuentes para investigar y restaurar. Seleccionar aptas no corrige ni autoriza descartar las ocho filas rechazadas del lote original.

**Salida esperada:** Comparación de los cuatro importes: solo `v01` cambia a cero en la copia.

**Para conversar:** ¿Qué tasa de rechazo esperas si falla una de cuatro filas?


In [5]:
subconjunto = crudos[crudos["fila_fuente"].isin(aceptadas["fila_fuente"])].copy()
degradado = subconjunto.copy(deep=True)
fila_alterada = degradado.index[0]
degradado.loc[fila_alterada, "monto"] = "0.00"
display(pd.DataFrame({
    "fila_fuente": subconjunto["fila_fuente"],
    "monto_preservado": subconjunto["monto"],
    "monto_degradado": degradado["monto"],
}))
assert len(subconjunto) == 4
assert subconjunto.loc[fila_alterada, "monto"] == "1500.00"
assert degradado.loc[fila_alterada, "monto"] == "0.00"
pd.testing.assert_frame_equal(crudos, crudos_preservados)


,fila_fuente,monto_preservado,monto_degradado
0,v01,1500.00,0.00
7,v08,700.00,700.00
8,v09,800.00,800.00
10,v11,1000.00,1000.00


## 6. Fijar la puerta de publicación

La tasa de rechazo es un **SLI**. La meta del contrato es un **SLO**: como máximo 20% en este ejercicio. Un SLA requiere un acuerdo y consecuencias. La puerta también bloquea lotes vacíos y totalmente rechazados. La huella identifica contenido, no el intento; cada intento tendrá su propio UUID.

**Salida esperada:** El escenario apto pasa; el original y el degradado no pasan. El límite exacto del 20% pasa.

**Para conversar:** ¿Por qué subir el umbral al 25% no arreglaría el importe cero?


In [6]:
def puerta_publicacion(total, rechazadas, max_rechazo=0.20):
    if not 0 <= max_rechazo <= 1 or total < 0 or not 0 <= rechazadas <= total:
        raise ValueError('Parametros invalidos para la puerta de publicacion')
    return total > 0 and rechazadas < total and rechazadas / total <= max_rechazo



def huella(datos):
    contenido = datos.to_json(orient='split', date_format='iso', index=False)
    return hashlib.sha256(contenido.encode('utf-8')).hexdigest()

MAX_RECHAZO = 0.20
puertas = pd.DataFrame([
    {"escenario": "apto", "total": 4, "rechazadas": 0},
    {"escenario": "original_defectuoso", "total": 12, "rechazadas": 8},
    {"escenario": "degradado", "total": 4, "rechazadas": 1},
])
puertas["publicable"] = [
    puerta_publicacion(f.total, f.rechazadas, MAX_RECHAZO)
    for f in puertas.itertuples(index=False)
]
display(puertas)
assert puertas["publicable"].tolist() == [True, False, False]
assert puerta_publicacion(5, 1, MAX_RECHAZO)
assert not puerta_publicacion(0, 0, MAX_RECHAZO)
print("Huella del subconjunto:", huella(subconjunto))


,escenario,total,rechazadas,publicable
0,apto,4,0,True
1,original_defectuoso,12,8,False
2,degradado,4,1,False


Huella del subconjunto: 5313c215882db9d5675e698b880ca6ce277951340747ded57e975c00f8d7ceab


## 7. Crear el destino y la auditoría

Creamos `productos`, `ventas`, `lotes` y `hallazgos`. Las restricciones protegen claves, referencias, importes positivos y monedas. `ventas` no guarda emails. La tabla `hallazgos` registra metadatos de fallas, **no una cuarentena durable con las filas completas**.

**Salida esperada:** Cuatro tablas creadas y cero ventas.

**Para conversar:** ¿Qué evidencia se perderá cuando cerremos esta base en memoria?


In [7]:
def nueva_conexion():
    conexion = sqlite3.connect(':memory:')
    conexion.execute('PRAGMA foreign_keys = ON')
    conexion.executescript('''
        CREATE TABLE productos (id_producto INTEGER PRIMARY KEY, producto TEXT NOT NULL);
        CREATE TABLE ventas (
            id_venta INTEGER PRIMARY KEY,
            id_producto INTEGER NOT NULL REFERENCES productos(id_producto),
            fecha_venta TEXT NOT NULL,
            monto_centavos INTEGER NOT NULL CHECK (monto_centavos > 0),
            moneda TEXT NOT NULL CHECK (moneda IN ('ARS', 'USD', 'EUR')),
            fila_fuente TEXT NOT NULL,
            lote_inicial TEXT NOT NULL
        );
        CREATE TABLE lotes (
            lote TEXT PRIMARY KEY, huella TEXT NOT NULL, contrato TEXT NOT NULL,
            estado TEXT NOT NULL, total INTEGER NOT NULL, aceptadas INTEGER NOT NULL,
            rechazadas INTEGER NOT NULL, nuevas INTEGER NOT NULL, existentes INTEGER NOT NULL
        );
        CREATE TABLE hallazgos (
            lote TEXT NOT NULL REFERENCES lotes(lote), fila_fuente TEXT NOT NULL,
            regla TEXT NOT NULL, accion TEXT NOT NULL
        );
    ''')
    return conexion

conexion = nueva_conexion()
display(pd.read_sql_query(
    "SELECT name FROM sqlite_master WHERE type = 'table' ORDER BY name",
    conexion,
))
display(pd.read_sql_query("PRAGMA table_info(ventas)", conexion))
assert conexion.execute("SELECT COUNT(*) FROM ventas").fetchone()[0] == 0
print("Destino vacío; no se ha publicado ningún escenario.")


,name
0,hallazgos
1,lotes
2,productos
3,ventas


,cid,name,type,notnull,dflt_value,pk
0,0,id_venta,INTEGER,0,None,1
1,1,id_producto,INTEGER,1,None,0
2,2,fecha_venta,TEXT,1,None,0
3,3,monto_centavos,INTEGER,1,None,0
4,4,moneda,TEXT,1,None,0
5,5,fila_fuente,TEXT,1,None,0
6,6,lote_inicial,TEXT,1,None,0


Destino vacío; no se ha publicado ningún escenario.


## 8. Definir el ETL que vamos a monitorear

Lee el recorrido de la función: normalizar → evaluar → decidir → cargar en transacción → reconciliar por moneda → registrar evidencia. La validación de esquema ocurre **antes** de crear el ID de lote: por eso más adelante registraremos la deriva en un evento de orquestación separado.

Una venta idéntica cuenta como existente; un contenido diferente produce error y rollback. Las diferencias de reconciliación también lanzan errores explícitos. El bloque de excepción conserva la auditoría y **relanza** el error, no lo oculta. La etiqueta heredada `ERROR_TECNICO` no demuestra que un conflicto sea transitorio.

**Salida esperada:** Mensaje de preparación: definir una función todavía no publica datos.

**Para conversar:** ¿Qué diferencia hay entre un lote bloqueado por calidad y un error de ejecución?


In [8]:
def ejecutar_etl(datos_crudos, productos, referencia, conexion):
    datos = normalizar(datos_crudos)
    aceptadas, rechazadas, metricas, detalle = evaluar(datos, productos, referencia)
    lote = uuid4().hex
    estado = 'PUBLICADO' if puerta_publicacion(len(datos), len(rechazadas), MAX_RECHAZO) else 'BLOQUEADO'
    nuevas, existentes = 0, 0
    try:
        with conexion:
            if estado == 'PUBLICADO':
                for producto in productos.itertuples(index=False):
                    existente = conexion.execute('SELECT producto FROM productos WHERE id_producto=?', (int(producto.id_producto),)).fetchone()
                    if existente is None:
                        conexion.execute('INSERT INTO productos VALUES (?, ?)', (int(producto.id_producto), producto.producto))
                    elif existente[0] != producto.producto:
                        raise ValueError('Cambio de maestro: requiere politica SCD aprobada')
                for venta in aceptadas.itertuples(index=False):
                    valores = (int(venta.id_producto), venta.fecha_venta.date().isoformat(), int(venta.monto_centavos), str(venta.moneda))
                    existente = conexion.execute('SELECT id_producto, fecha_venta, monto_centavos, moneda FROM ventas WHERE id_venta=?', (int(venta.id_venta),)).fetchone()
                    if existente is None:
                        conexion.execute('INSERT INTO ventas VALUES (?, ?, ?, ?, ?, ?, ?)', (int(venta.id_venta), *valores, venta.fila_fuente, lote))
                        nuevas += 1
                    elif existente == valores:
                        existentes += 1
                    else:
                        raise ValueError('Conflicto de contenido para una venta ya publicada')
                claves = [int(identificador) for identificador in aceptadas['id_venta']]
                marcadores = ','.join('?' for _ in claves)
                destino = pd.read_sql_query(f'SELECT id_venta, moneda, monto_centavos FROM ventas WHERE id_venta IN ({marcadores})', conexion, params=claves)
                if len(destino) != len(aceptadas):
                    raise ValueError('Reconciliacion fallida: cantidad de ventas')
                esperado = {str(moneda): int(total) for moneda, total in aceptadas.groupby('moneda')['monto_centavos'].sum().items()}
                obtenido = {str(moneda): int(total) for moneda, total in destino.groupby('moneda')['monto_centavos'].sum().items()}
                if esperado != obtenido:
                    raise ValueError('Reconciliacion fallida: totales por moneda')
            conexion.execute('INSERT INTO lotes VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)', (lote, huella(datos_crudos), VERSION_CONTRATO, estado, len(datos), len(aceptadas), len(rechazadas), nuevas, existentes))
            conexion.executemany('INSERT INTO hallazgos VALUES (?, ?, ?, ?)', [(lote, fila.fila_fuente, fila.regla, fila.accion) for fila in detalle.itertuples(index=False)])
    except Exception:
        with conexion:
            conexion.execute('INSERT INTO lotes VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)', (lote, huella(datos_crudos), VERSION_CONTRATO, 'ERROR_TECNICO', len(datos), len(aceptadas), len(rechazadas), 0, 0))
        raise
    return {'lote': lote, 'estado': estado, 'nuevas': nuevas, 'existentes': existentes, 'aceptadas': aceptadas, 'rechazadas': rechazadas, 'metricas': metricas, 'detalle': detalle}

print("ETL definido: puerta, transacción, reconciliación y auditoría.")
print("Sigue habiendo", conexion.execute("SELECT COUNT(*) FROM ventas").fetchone()[0], "ventas.")


ETL definido: puerta, transacción, reconciliación y auditoría.
Sigue habiendo 0 ventas.


## 9. Publicar el escenario apto

Ejecutamos únicamente el subconjunto apto y miramos el destino de inmediato. La publicación usa las mismas reglas; no implica que cada fila esté libre de advertencias.

**Salida esperada:** PUBLICADO, 4 ventas nuevas y 0 existentes; totales ARS 330000 y USD 70000 centavos.

**Para conversar:** ¿Por qué reconciliamos por moneda en lugar de calcular un total único?


In [9]:
resultado_apto = ejecutar_etl(subconjunto, productos, referencia, conexion)
print("Estado:", resultado_apto["estado"])
print("Nuevas:", resultado_apto["nuevas"], "| Existentes:", resultado_apto["existentes"])
ventas_publicadas = pd.read_sql_query("SELECT * FROM ventas ORDER BY id_venta", conexion)
display(ventas_publicadas)
display(ventas_publicadas.groupby("moneda", as_index=False)["monto_centavos"].sum())
assert resultado_apto["estado"] == "PUBLICADO"
assert (resultado_apto["nuevas"], resultado_apto["existentes"]) == (4, 0)
assert ventas_publicadas.groupby("moneda")["monto_centavos"].sum().to_dict() == {
    "ARS": 330000, "USD": 70000,
}
assert "email" not in ventas_publicadas.columns


Estado: PUBLICADO
Nuevas: 4 | Existentes: 0


,id_venta,id_producto,fecha_venta,monto_centavos,moneda,fila_fuente,lote_inicial
0,1,101,2026-10-05,150000,ARS,v01,d9c44f79a7d7456a965e34dcaf69d1f7
1,7,102,2026-10-05,70000,USD,v08,d9c44f79a7d7456a965e34dcaf69d1f7
2,8,102,2026-09-30,80000,ARS,v09,d9c44f79a7d7456a965e34dcaf69d1f7
3,10,101,2026-10-05,100000,ARS,v11,d9c44f79a7d7456a965e34dcaf69d1f7


,moneda,monto_centavos
0,ARS,330000
1,USD,70000


## 10. Observar el bloqueo del lote original

El lote completo tiene ocho rechazos de doce. La puerta bloquea toda esa publicación y conserva auditoría y hallazgos. Las cuatro ventas publicadas en el paso anterior no deben cambiar.

**Salida esperada:** BLOQUEADO, 8 de 12 rechazadas y 17 hallazgos; destino intacto.

**Para conversar:** ¿Por qué 17 hallazgos no significan 17 filas rechazadas?


In [10]:
resultado_original = ejecutar_etl(crudos, productos, referencia, conexion)
print("Estado:", resultado_original["estado"])
print("Rechazadas:", len(resultado_original["rechazadas"]), "de", len(crudos))
display(resultado_original["detalle"])
assert resultado_original["estado"] == "BLOQUEADO"
pd.testing.assert_frame_equal(
    pd.read_sql_query("SELECT * FROM ventas ORDER BY id_venta", conexion),
    ventas_publicadas,
)
print("El destino conserva exactamente las cuatro ventas previas.")


Estado:

 BLOQUEADO
Rechazadas: 8 de 12


,fila_fuente,regla,accion
0,v12,DQ_ID,cuarentena
1,v02,DQ_UNICIDAD,cuarentena
2,v03,DQ_UNICIDAD,cuarentena
3,v12,DQ_UNICIDAD,cuarentena
4,v04,DQ_FECHA,cuarentena
5,v10,DQ_FECHA,cuarentena
6,v05,DQ_MONTO,cuarentena
7,v06,DQ_MONEDA,cuarentena
8,v07,DQ_FK_PRODUCTO,cuarentena
9,v04,DQ_ACTUALIZACION,cuarentena


El destino conserva exactamente las cuatro ventas previas.


## 11. Observar el escenario degradado

Ejecutamos ahora la copia con importe cero. El denominador es cuatro, no doce. Una sola fila rechazada alcanza 25%, supera el 20% y bloquea el lote. No se sobrescribe la venta ya publicada con un valor inválido.

**Salida esperada:** BLOQUEADO y 1 de 4 rechazada; el destino sigue intacto.

**Para conversar:** ¿Una tasa mayor implica necesariamente más filas defectuosas en términos absolutos?


In [11]:
resultado_degradado = ejecutar_etl(degradado, productos, referencia, conexion)
tasa_degradado = len(resultado_degradado["rechazadas"]) / len(degradado)
print("Estado:", resultado_degradado["estado"])
print(f"Rechazo: {len(resultado_degradado['rechazadas'])}/{len(degradado)} = {tasa_degradado:.0%}")
display(resultado_degradado["detalle"])
assert resultado_degradado["estado"] == "BLOQUEADO"
assert tasa_degradado == 0.25
pd.testing.assert_frame_equal(
    pd.read_sql_query("SELECT * FROM ventas ORDER BY id_venta", conexion),
    ventas_publicadas,
)
print("Bloqueo verificado: ninguna venta publicada se alteró.")


Estado: BLOQUEADO
Rechazo: 1/4 = 25%


,fila_fuente,regla,accion
0,v01,DQ_MONTO,cuarentena
1,v08,DQ_EMAIL,advertencia
2,v09,DQ_EMAIL,advertencia
3,v09,DQ_VIGENCIA,advertencia
4,v01,DQ_EXACTITUD,advertencia
5,v11,DQ_EXACTITUD,advertencia


Bloqueo verificado: ninguna venta publicada se alteró.


## 12. Construir el tablero con denominadores visibles

Reunimos resultados **ya ejecutados**, sin volver a cargar los escenarios. Rechazo = filas rechazadas / total. Cobertura de exactitud = evaluables / total. Cumplimiento de exactitud = evaluables correctas / evaluables. Si no hay evaluables, el cumplimiento no es 100%: queda sin valor. Una advertencia de exactitud no cambia por sí sola la puerta de este contrato.

**Salida esperada:** PUBLICADO, BLOQUEADO y BLOQUEADO, con denominadores y métricas diferenciadas.

**Para conversar:** ¿Por qué el subconjunto apto puede tener 50% de cumplimiento de exactitud?


In [12]:
monitoreo = []
escenarios = [
    ("apto", subconjunto, resultado_apto),
    ("original_defectuoso", crudos, resultado_original),
    ("degradado", degradado, resultado_degradado),
]
for nombre, fuente, resultado in escenarios:
    exactitud = resultado["metricas"].set_index("regla").loc["DQ_EXACTITUD"]
    total = len(fuente)
    evaluables = int(exactitud["evaluados"])
    monitoreo.append({
        "escenario": nombre,
        "lote": resultado["lote"],
        "estado": resultado["estado"],
        "total_filas": total,
        "rechazadas": len(resultado["rechazadas"]),
        "tasa_rechazo": len(resultado["rechazadas"]) / total if total else None,
        "hallazgos": len(resultado["detalle"]),
        "evaluables_exactitud": evaluables,
        "fallidos_exactitud": int(exactitud["fallidos"]),
        "cobertura_exactitud_pct": 100 * evaluables / total if total else None,
        "cumplimiento_exactitud_pct": exactitud["cumplimiento_pct"],
        "contrato": VERSION_CONTRATO,
    })
tablero = pd.DataFrame(monitoreo)
display(tablero)
assert tablero["estado"].tolist() == ["PUBLICADO", "BLOQUEADO", "BLOQUEADO"]
assert tablero["rechazadas"].tolist() == [0, 8, 1]
assert tablero["evaluables_exactitud"].tolist() == [2, 2, 2]
assert tablero["cumplimiento_exactitud_pct"].tolist() == [50.0, 50.0, 0.0]
assert conexion.execute("SELECT COUNT(*) FROM lotes").fetchone()[0] == 3


,escenario,lote,estado,total_filas,rechazadas,tasa_rechazo,hallazgos,evaluables_exactitud,fallidos_exactitud,cobertura_exactitud_pct,cumplimiento_exactitud_pct,contrato
0,apto,d9c44f79a7d7456a965e34dcaf69d1f7,PUBLICADO,4,0,0.000000,4,2,1,50.000000,50.0,1.0.0
1,original_defectuoso,da95eba5322f4b9ca0aab01854890c49,BLOQUEADO,12,8,0.666667,17,2,1,16.666667,50.0,1.0.0
2,degradado,4b418751608b42be9fcd51536ee8cf8a,BLOQUEADO,4,1,0.250000,6,2,2,50.000000,0.0,1.0.0


## 13. Detectar deriva antes de transformar

Quitamos `moneda` solo de una copia. El preflight debe detenerla y no crear un lote en SQLite. Conservamos un evento con ID propio, mensaje y responsable. Este evento vive en memoria: en producción necesita persistencia, correlación con el intento del orquestador y notificación real.

**Salida esperada:** Un evento BLOQUEADO_PREFLIGHT; siguen existiendo 3 lotes y 4 ventas.

**Para conversar:** ¿Por qué el tablero de lotes por sí solo no mostraría esta falla?


In [13]:
eventos_orquestacion = []
lotes_antes_deriva = conexion.execute("SELECT COUNT(*) FROM lotes").fetchone()[0]
try:
    ejecutar_etl(subconjunto.drop(columns=["moneda"]), productos, referencia, conexion)
except ValueError as error:
    if "Deriva de esquema" not in str(error):
        raise
    eventos_orquestacion.append({
        "id_evento": uuid4().hex,
        "tipo": "DERIVA_ESQUEMA",
        "estado": "BLOQUEADO_PREFLIGHT",
        "mensaje": str(error),
        "responsable": "Steward Comercial",
        "contrato": VERSION_CONTRATO,
    })
    print("Bloqueo esperado:", error)
assert len(eventos_orquestacion) == 1, "La deriva debería bloquearse."
assert conexion.execute("SELECT COUNT(*) FROM lotes").fetchone()[0] == lotes_antes_deriva
pd.testing.assert_frame_equal(
    pd.read_sql_query("SELECT * FROM ventas ORDER BY id_venta", conexion),
    ventas_publicadas,
)
display(pd.DataFrame(eventos_orquestacion))


Bloqueo esperado: Deriva de esquema: revisar contrato antes de transformar


,id_evento,tipo,estado,mensaje,responsable,contrato
0,887f004727724d0e88927b6441c0439b,DERIVA_ESQUEMA,BLOQUEADO_PREFLIGHT,Deriva de esquema: revisar contrato antes de t...,Steward Comercial,1.0.0


## 14. Abrir y asignar el incidente de calidad

Este incidente corresponde al **lote degradado**, no al evento separado de deriva. Registramos causa conocida, acción, responsable y vínculo al lote. La autorización es simulada: un texto no es una firma ni un flujo real de aprobación.

**Salida esperada:** INC-DEMO-001 en ASIGNADO, vinculado al lote degradado.

**Para conversar:** ¿Quién investiga y quién debe autorizar una corrección en un proceso real?


In [14]:
def json_seguro(objeto):
    return json.dumps(objeto, ensure_ascii=True, indent=2, allow_nan=False)

incidente = {
    "id": "INC-DEMO-001",
    "lote_origen": resultado_degradado["lote"],
    "estado": "ASIGNADO",
    "regla": "DQ_MONTO",
    "severidad": "alta para publicacion del lote",
    "responsable": "Steward Comercial",
    "causa": "importe alterado a cero en una copia de prueba",
    "accion": "restaurar desde la fuente ficticia preservada y revalidar",
    "autorizacion": "simulada; no representa firma ni autenticacion",
}
historial_incidente = [{
    "paso": 1, "estado": incidente["estado"], "evidencia": incidente["lote_origen"],
}]
print(json_seguro(incidente))
assert incidente["lote_origen"] == tablero.loc[2, "lote"]


{
  "id": "INC-DEMO-001",
  "lote_origen": "4b418751608b42be9fcd51536ee8cf8a",
  "estado": "ASIGNADO",
  "regla": "DQ_MONTO",
  "severidad": "alta para publicacion del lote",
  "responsable": "Steward Comercial",
  "causa": "importe alterado a cero en una copia de prueba",
  "accion": "restaurar desde la fuente ficticia preservada y revalidar",
  "autorizacion": "simulada; no representa firma ni autenticacion"
}


## 15. Restaurar desde evidencia preservada

Restauramos el subconjunto completo desde la copia conocida, sin tocar el degradado. El incidente todavía **no está cerrado**: una corrección propuesta debe pasar por las mismas reglas y la misma puerta.

**Salida esperada:** Importe cero en el degradado y 1500.00 en la fuente recuperada; estado pendiente de validación.

**Para conversar:** ¿Por qué copiar un valor sin evidencia o cambiar el umbral sería una mala recuperación?


In [15]:
fuente_recuperada = subconjunto.copy(deep=True)
incidente["estado"] = "CORREGIDO_PENDIENTE_VALIDACION"
historial_incidente.append({
    "paso": 2, "estado": incidente["estado"],
    "evidencia": "copia del subconjunto ficticio preservado",
})
display(pd.DataFrame({
    "fila_fuente": subconjunto["fila_fuente"],
    "monto_degradado": degradado["monto"],
    "monto_recuperado": fuente_recuperada["monto"],
}))
assert degradado.loc[fila_alterada, "monto"] == "0.00"
assert fuente_recuperada.loc[fila_alterada, "monto"] == "1500.00"
assert MAX_RECHAZO == 0.20
print("Estado:", incidente["estado"])


,fila_fuente,monto_degradado,monto_recuperado
0,v01,0.00,1500.00
7,v08,700.00,700.00
8,v09,800.00,800.00
10,v11,1000.00,1000.00


Estado: CORREGIDO_PENDIENTE_VALIDACION


## 16. Reintentar y verificar el destino

Ejecutamos el ETL sobre la fuente recuperada. Como las cuatro ventas correctas ya existen, esperamos cero nuevas y cuatro existentes. Verificamos además que el contenido completo del destino sea idéntico al publicado inicialmente.

**Salida esperada:** PUBLICADO, 0 nuevas, 4 existentes; mismo destino y mismo umbral.

**Para conversar:** ¿Por qué una recuperación publicada no necesita insertar nuevas ventas?


In [16]:
recuperacion = ejecutar_etl(fuente_recuperada, productos, referencia, conexion)
assert recuperacion["estado"] == "PUBLICADO"
assert (recuperacion["nuevas"], recuperacion["existentes"]) == (0, 4)
pd.testing.assert_frame_equal(
    pd.read_sql_query("SELECT * FROM ventas ORDER BY id_venta", conexion),
    ventas_publicadas,
)
assert MAX_RECHAZO == 0.20
assert huella(fuente_recuperada) == huella(subconjunto)
assert recuperacion["lote"] != resultado_apto["lote"]
print("Estado:", recuperacion["estado"])
print("Nuevas:", recuperacion["nuevas"], "| Existentes:", recuperacion["existentes"])
print("Destino idéntico, nuevo ID de intento y umbral sin cambios.")
display(pd.read_sql_query("SELECT * FROM ventas ORDER BY id_venta", conexion))


Estado: PUBLICADO
Nuevas: 0 | Existentes: 4
Destino idéntico, nuevo ID de intento y umbral sin cambios.


,id_venta,id_producto,fecha_venta,monto_centavos,moneda,fila_fuente,lote_inicial
0,1,101,2026-10-05,150000,ARS,v01,d9c44f79a7d7456a965e34dcaf69d1f7
1,7,102,2026-10-05,70000,USD,v08,d9c44f79a7d7456a965e34dcaf69d1f7
2,8,102,2026-09-30,80000,ARS,v09,d9c44f79a7d7456a965e34dcaf69d1f7
3,10,101,2026-10-05,100000,ARS,v11,d9c44f79a7d7456a965e34dcaf69d1f7


## 17. Cerrar el incidente con evidencia

Solo después de verificar el reintento y el destino declaramos el cierre. Guardamos el vínculo al intento de recuperación y mostramos las tres transiciones. El evento de deriva sigue siendo una evidencia separada; este cierre no lo resuelve ni corrige las ocho filas defectuosas del lote original.

**Salida esperada:** CERRADO_VERIFICADO y 4 intentos en auditoría; el último tiene 0 nuevas y 4 existentes.

**Para conversar:** ¿Qué pruebas permiten cerrar el incidente y cuáles son solo declaraciones?


In [17]:
incidente["lote_recuperacion"] = recuperacion["lote"]
incidente["estado"] = "CERRADO_VERIFICADO"
historial_incidente.append({
    "paso": 3, "estado": incidente["estado"], "evidencia": recuperacion["lote"],
})
print(json_seguro(incidente))
display(pd.DataFrame(historial_incidente))
auditoria = pd.read_sql_query(
    "SELECT lote, estado, total, rechazadas, nuevas, existentes FROM lotes ORDER BY rowid",
    conexion,
)
display(auditoria)
assert auditoria["estado"].tolist() == [
    "PUBLICADO", "BLOQUEADO", "BLOQUEADO", "PUBLICADO",
]
assert historial_incidente[-1]["estado"] == "CERRADO_VERIFICADO"
assert incidente["lote_origen"] != incidente["lote_recuperacion"]


{
  "id": "INC-DEMO-001",
  "lote_origen": "4b418751608b42be9fcd51536ee8cf8a",
  "estado": "CERRADO_VERIFICADO",
  "regla": "DQ_MONTO",
  "severidad": "alta para publicacion del lote",
  "responsable": "Steward Comercial",
  "causa": "importe alterado a cero en una copia de prueba",
  "accion": "restaurar desde la fuente ficticia preservada y revalidar",
  "autorizacion": "simulada; no representa firma ni autenticacion",
  "lote_recuperacion": "15ef90395d00448d87e8a7eac538ca9a"
}


,paso,estado,evidencia
0,1,ASIGNADO,4b418751608b42be9fcd51536ee8cf8a
1,2,CORREGIDO_PENDIENTE_VALIDACION,copia del subconjunto ficticio preservado
2,3,CERRADO_VERIFICADO,15ef90395d00448d87e8a7eac538ca9a


,lote,estado,total,rechazadas,nuevas,existentes
0,d9c44f79a7d7456a965e34dcaf69d1f7,PUBLICADO,4,0,4,0
1,da95eba5322f4b9ca0aab01854890c49,BLOQUEADO,12,8,0,0
2,4b418751608b42be9fcd51536ee8cf8a,BLOQUEADO,4,1,0,0
3,15ef90395d00448d87e8a7eac538ca9a,PUBLICADO,4,0,0,4


## 18. Cerrar la conexión

Comprobamos una vez más que las fuentes se conservaron y cerramos SQLite. Las tablas se pierden; las salidas del notebook quedan como evidencia didáctica, no como repositorio de auditoría operativo.

**Salida esperada:** Confirmación de cierre y recordatorio de los pendientes del lote original.

**Para conversar:** ¿Dónde guardarías incidentes, eventos y rechazos en un sistema real?


In [18]:
pd.testing.assert_frame_equal(crudos, crudos_preservados)
assert conexion.execute("SELECT COUNT(*) FROM ventas").fetchone()[0] == 4
conexion.close()
print("Conexión cerrada. Recuperación verificada sin relajar umbrales ni duplicar ventas.")
print("Las ocho filas defectuosas originales siguen pendientes de investigación.")
print("Alertas, aprobaciones y persistencia operativa no están implementadas en esta demo.")


Conexión cerrada. Recuperación verificada sin relajar umbrales ni duplicar ventas.
Las ocho filas defectuosas originales siguen pendientes de investigación.
Alertas, aprobaciones y persistencia operativa no están implementadas en esta demo.


## Síntesis y ejercicios para los alumnos

**Recorrido:** observar → comparar → bloquear → registrar → asignar →
restaurar con evidencia → revalidar → comprobar destino → cerrar.

1. Explica los denominadores de rechazo, cobertura de exactitud y cumplimiento de exactitud.
2. Propón una alerta con destinatario, severidad, plazo y criterio de escalamiento.
   No confundas registrarla en una lista con notificarla.
3. Diseña cómo persistir el evento de preflight y las transiciones del incidente.
4. En una nueva ejecución completa, experimenta con dos importes cero y predice el rechazo.
   No cambies el umbral para conseguir una publicación.
5. Propón un cambio de esquema con nueva versión de contrato, aprobación y compatibilidad.

**Resultado esperado:** tres escenarios iniciales PUBLICADO / BLOQUEADO / BLOQUEADO;
deriva detectada antes de crear lote; recuperación PUBLICADO con cero nuevas y
cuatro existentes; incidente CERRADO_VERIFICADO.

**Límites:** no hay persistencia operativa, notificaciones reales, firma del Owner,
IAM ni cuarentena durable de filas. No se ha solucionado el lote original de 12.
